# Re-run detection on an existing registered binary

Workflow: run `lsp.pipeline` once to register + detect. Then point `lsp.run_plane` at the resulting `data.bin` and re-run *detection only*, optionally with different params (different `diameter`, `cellprob_threshold`, etc.). No re-registration.

Why this works: when `input_data` is a `.bin` file inside a directory that also contains an `ops.npy`, `lsp.run_plane` recognizes that as a staged plane-dir, sets `skip_imwrite=True`, and processes in place. With `force_detect=True` and no `force_reg`, suite2p runs detection + extraction against the existing registered binary.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import matplotlib.pyplot as plt

import mbo_utilities as mbo
import lbm_suite2p_python as lsp

# input — a 4D TZYX tiff produced by mbo.imwrite (or any source lsp.pipeline accepts)
tiff_dir = Path(r"D:/demo/bidi_compare/tiff_mbo_corrected")  # change to your input
save_path = Path(r"D:/demo/redetect_demo")
save_path.mkdir(parents=True, exist_ok=True)

PLANE = 7  # 1-based plane to demo on

## 1. Initial pipeline: register + detect

`keep_reg=True` (the default) preserves `data.bin` so we can re-detect later. `keep_raw=True` lets you also re-register later if you want — leave it `False` if you only plan to re-detect.

In [ ]:
ops_initial = lsp.default_ops()
ops_initial["diameter"] = 6
ops_initial["anatomical_only"] = 3

lsp.pipeline(
    input_data=tiff_dir,
    save_path=save_path,
    ops=ops_initial,
    planes=[PLANE],
    keep_reg=True,        # preserve data.bin
    keep_raw=False,       # set True if you also want to re-register later
    force_reg=True,       # ignore any cached output for this demo
    force_detect=True,
)

## 2. Locate the plane dir

`lsp.pipeline` writes `<save_path>/zplane<NN>_tp.../{data.bin, ops.npy, stat.npy, F.npy, ...}`.

In [ ]:
plane_dirs = sorted(save_path.glob(f"zplane{PLANE:02d}_*"))
assert plane_dirs, f"no plane{PLANE} output under {save_path}"
plane_dir = plane_dirs[0]
print("plane dir:", plane_dir)
for f in ("ops.npy", "data.bin", "data_raw.bin", "stat.npy", "F.npy", "iscell.npy"):
    p = plane_dir / f
    print(f"  {f:14s} {'OK' if p.exists() else 'MISSING'}  {p.stat().st_size if p.exists() else 0:>14,d} bytes")

ops0 = np.load(plane_dir/"ops.npy", allow_pickle=True).item()
stat0 = np.load(plane_dir/"stat.npy", allow_pickle=True)
iscell0 = np.load(plane_dir/"iscell.npy", allow_pickle=True)[:, 0].astype(bool)
print(f"\ninitial detection: {len(stat0)} ROIs total, {iscell0.sum()} accepted")
print(f"initial diameter:  {ops0.get('diameter')}")
print(f"Ly,Lx:             {ops0.get('Ly')}, {ops0.get('Lx')}")
print(f"nframes:           {ops0.get('nframes_chan1') or ops0.get('nframes')}")

## 3. Snapshot the first detection

`run_plane` overwrites `stat.npy`, `F.npy`, `iscell.npy`, etc. in the plane dir. Copy them aside so we can compare before/after.

In [ ]:
snap_dir = plane_dir / "_snapshot_run1"
snap_dir.mkdir(exist_ok=True)
for fname in ("ops.npy", "stat.npy", "F.npy", "Fneu.npy", "iscell.npy", "spks.npy"):
    src = plane_dir / fname
    if src.exists():
        shutil.copy2(src, snap_dir / fname)
print("snapshotted run1 outputs to", snap_dir)

## 4. Re-run detection only on the same data.bin

Key invocation:
- `input_data = plane_dir / "data.bin"` — points at the existing registered binary; because `ops.npy` is in the same dir, `run_plane` recognizes this as a staged plane-dir.
- `save_path = plane_dir.parent` — same parent so the in-place branch fires (`run_plane` checks `input_path.parent == save_path`).
- `force_detect=True` — re-run detection even though `stat.npy` already exists.
- `force_reg=False` — leave registration alone (and even if it weren't, missing `data_raw.bin` would silently downgrade).
- ops only carries detection params — registration knobs are read from the existing `ops.npy`.

In [ ]:
ops_redetect = {
    "diameter": 4,                  # different from run1
    "anatomical_only": 4,
    "cellprob_threshold": -2.0,
    "do_registration": 0,           # explicit: skip registration
    "roidetect": 1,
}

lsp.run_plane(
    input_data=plane_dir / "data.bin",
    save_path=plane_dir.parent,
    ops=ops_redetect,
    force_reg=False,
    force_detect=True,
    keep_reg=True,                  # don't blow away data.bin on the way out
)

## 5. Compare run1 vs run2

In [ ]:
stat1 = np.load(snap_dir/"stat.npy", allow_pickle=True)
isc1  = np.load(snap_dir/"iscell.npy", allow_pickle=True)[:,0].astype(bool)
stat2 = np.load(plane_dir/"stat.npy", allow_pickle=True)
isc2  = np.load(plane_dir/"iscell.npy", allow_pickle=True)[:,0].astype(bool)
ops2  = np.load(plane_dir/"ops.npy",   allow_pickle=True).item()

print(f"run1: {len(stat1):4d} total, {isc1.sum():4d} accepted   (diameter={ops0.get('diameter')})")
print(f"run2: {len(stat2):4d} total, {isc2.sum():4d} accepted   (diameter={ops2.get('diameter')})")

def centroids(stat, isc):
    return np.array([[s['med'][0], s['med'][1]] for s, ok in zip(stat, isc) if ok])

c1 = centroids(stat1, isc1)
c2 = centroids(stat2, isc2)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, c, title in zip(ax, (c1, c2), ("run1", "run2")):
    a.imshow(ops2['meanImg'], cmap='gray')
    a.scatter(c[:,1], c[:,0], s=8, c='red', alpha=0.7)
    a.set_title(f"{title}: {len(c)} accepted ROIs")
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### Notes

- **The registered binary is unchanged across the two runs** — both detections are evaluated against the exact same pixels. You can re-run cell 4 with any other detection knob (`cellprob_threshold`, `flow_threshold`, `min_neuropil_pixels`, etc.) to scan parameter space without paying registration cost.
- **If `force_detect=True` doesn't appear to do anything**, check that:
  - `input_data` ends in `data.bin` (not the plane-dir itself or a tiff path).
  - `save_path` is `plane_dir.parent`.
  - The plane dir contains both `data.bin` and `ops.npy`.
- **To re-register**, you also need `data_raw.bin`. That file is deleted at the end of every run unless `keep_raw=True` was set. Without it, `force_reg=True` is silently downgraded to `do_registration=0` inside `run_plane_bin` (see `lbm_suite2p_python/run_lsp.py:1456`).